# M6.1 Flux, Motion and the Induced Emf

Book pointer: chapter 14, sections 14.2 (motional emf), 14.3 (Faraday's law) and 14.4 (Faraday's law on differential form).

## 1. Recall

Answer on paper before running any cell.

1. State Faraday's flux rule for a closed loop. What sets the sign of the flux, and what sets the direction in which the emf is counted? State Lenz's law and say what it adds to the formula.
2. A loop moves through a static field with velocity $\mathbf{v}$. Write the motional emf as a line integral. Which force on the charges in the wire does it come from?
3. Why is the flux of $\mathbf{B}$ the same through every surface bounded by the same loop? Which of Maxwell's equations makes that true, and how does it let you write the flux as $\oint\mathbf{A}\cdot d\mathbf{l}$?
4. Write Faraday's law in differential form. What does it say about the electric field when $\mathbf{B}$ changes in time, and why can that field not be written as $-\nabla V$?
5. You have the flux at equally spaced times $t_k$. Write the central-difference estimate of $d\Phi/dt$ at $t_k$ and say how its error scales with the time step.

## 2. Principle to algorithm

Faraday's law says the emf around a closed loop is

$$\mathcal{E} = \oint_C (\mathbf{E} + \mathbf{v}\times\mathbf{B})\cdot d\mathbf{l} = -\frac{d\Phi}{dt}, \qquad \Phi = \int_S \mathbf{B}\cdot d\mathbf{S},$$

where $\mathbf{v}$ is the velocity of each piece of the wire and $S$ is any surface bounded by $C$. The loop direction and the surface normal are tied by the right-hand rule: counter-clockwise about $\hat{\mathbf{n}}$. Two physically different situations satisfy the same rule. If the loop moves through a static field, $\mathbf{E} = 0$ and the emf is the magnetic force on the moving charges, the motional emf $\oint(\mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$. If the field changes in time and the loop is still, $\mathbf{v} = 0$ and the emf comes from an induced electric field with $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$. The flux rule covers both, and which description applies depends on the frame.

That gives two independent numerical routes to the same number, and the notebook uses both as a check on each other.

1. **Flux route.** Sample the loop's position at times $t_k$. At each time compute the flux, with `induction.disk_flux` (a polar midpoint rule over the flat disk spanning the loop) or, when the disk would pass through a singular source, as $\oint\mathbf{A}\cdot d\mathbf{l}$ around the rim. Then `induction.emf_from_flux` takes $-d\Phi/dt$ by central differences.
2. **Motional route.** At one instant, cut the loop into segments with `biot_savart.segments`, evaluate $\mathbf{v}\times\mathbf{B}$ at each midpoint, dot it with $d\mathbf{l}$ and sum. No time derivative is involved.

The error sources are different for the two routes, which is what makes the comparison useful. The flux route is limited by the time step, because a central difference has an error of order $\Delta t^2\,\Phi'''$, and by the quadrature of the flux, whose small errors the differentiation amplifies by $1/\Delta t$. The motional route is limited by the polyline: segments must be short compared with the distance to any source of field. Both inherit the error of the field itself, here a Biot-Savart sum (`biot_savart.wire_B`), a point dipole (`biot_savart.dipole_B`) or an exact formula.

### Animation

A bar magnet moves along the axis of a ring, through it and out the other side. Watch the flux readout rise as the magnet approaches, peak as it passes through the ring, and fall as it leaves, while the emf plot below it traces the slope of that curve with a minus sign. The induced current on the ring turns one way while the flux grows and the other way while it shrinks: in both halves it makes a field that opposes the change, which is Lenz's law.

In [ ]:
from emlib import show
show.animation("m6_waves", "FaradayInduction")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import biot_savart as bs, induction as ind
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 4.5)
np.set_printoptions(precision=4, suppress=True)

### A loop moving along the axis of a current loop

A fixed loop of radius $a = 10$ cm carries $I = 10$ A, counter-clockwise about $+z$, cut into 200 segments. A second, smaller loop of radius $b = 5$ cm moves along the common axis at $v = 2$ m/s, from 20 cm below the fixed loop to 20 cm above it. Its normal is $+z$ throughout.

The flux through the moving loop is $\Phi(z) = M(z)\,I$, where $M$ is the mutual inductance of two coaxial loops, known exactly in terms of elliptic integrals (`induction.coaxial_loops_M`). That gives a third, exact value of the emf, $\mathcal{E} = -I\,v\,dM/dz$, against which both numerical routes are measured.

The first cell checks the time step. Even with exact flux values, the central difference needs a step small enough to resolve the shortest length over which $\Phi(z)$ changes. Here that is the gap $a - b = 5$ cm between the two wires when the loops are coplanar, and the emf changes sign there.

In [ ]:
a, b, I, v = 0.10, 0.05, 10.0, 2.0
M = lambda z: ind.coaxial_loops_M(a, b, z)
dMdz = lambda z: (M(z + 1e-7) - M(z - 1e-7)) / 2e-7
emf_exact = lambda z: -I * v * dMdz(z)

print("samples   dz [mm]   max error of -dPhi/dt from exact flux values (relative to max |emf|)")
for N in (41, 81, 161, 321):
    t = np.linspace(0.0, 0.2, N)
    z = -0.2 + v * t
    e = ind.emf_from_flux(t, I * M(z))
    ref = emf_exact(z)
    print(f"{N:5d}    {1000 * (z[1] - z[0]):6.2f}    {np.abs(e - ref)[1:-1].max() / np.abs(ref).max():.2e}")

Halving the step divides the error by four: the central difference is second order. With a step of 2 mm the differentiation error is near $10^{-3}$ of the peak emf, small enough to see the other error sources. The cell below runs both routes with that step, over a window from 20 cm below the fixed loop to 10 cm above it, so that the flux at the start and at the end differ (section 4 uses that).

In [ ]:
mid_a, dl_a = bs.segments(bs.loop(a, 200))
B_loop = lambda p: bs.wire_B(mid_a, dl_a, I, p)

t1 = np.linspace(0.0, 0.15, 151)
z1 = -0.2 + v * t1                                     # from 20 cm below to 10 cm above the fixed loop
flux1 = np.array([ind.disk_flux(B_loop, (0, 0, z), (0, 0, 1), b, nr=30, nphi=60) for z in z1])
emf1_flux = ind.emf_from_flux(t1, flux1)

def motional_sum(path, velocity, B_func):
    mid, dl = bs.segments(path)
    vel = velocity(mid) if callable(velocity) else np.broadcast_to(velocity, mid.shape)
    return float(np.sum(np.cross(vel, B_func(mid)) * dl))

emf1_mot = np.array([motional_sum(bs.loop(b, 200, centre=(0, 0, z)), np.array([0, 0, v]), B_loop) for z in z1])
emf1_ex = emf_exact(z1)

scale1 = np.abs(emf1_ex).max()
err1_flux = np.abs(emf1_flux - emf1_ex)[1:-1].max() / scale1
err1_mot = np.abs(emf1_mot - emf1_ex).max() / scale1
print(f"flux by quadrature vs M I:     max relative error {np.abs(flux1 / (I * M(z1)) - 1).max():.1e}")
print(f"peak |emf|: {1e6 * scale1:.2f} uV")
print(f"flux route vs exact:           {err1_flux:.1e} of the peak")
print(f"motional route vs exact:       {err1_mot:.1e} of the peak")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(z1 * 100, 1e9 * flux1, "C0")
ax[0].set_xlabel("z of the moving loop [cm]"); ax[0].set_ylabel("flux [nWb]"); ax[0].set_title("flux through the moving loop")
ax[1].plot(t1 * 1e3, 1e6 * emf1_ex, "k-", lw=3, alpha=0.3, label="exact, -I v dM/dz")
ax[1].plot(t1[1:-1] * 1e3, 1e6 * emf1_flux[1:-1], "C0--", label="flux route")
ax[1].plot(t1 * 1e3, 1e6 * emf1_mot, "C1:", lw=2, label="motional route")
ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("emf [uV]"); ax[1].legend(); ax[1].set_title("emf: two routes and the exact value")
plt.tight_layout(); plt.show()

The flux rises as the small loop approaches, peaks when the loops are coplanar, and falls as it leaves. The emf is minus the slope: negative on the way in, zero at the peak, positive on the way out. A negative emf means the induced current runs clockwise about $+z$, so its own field points down through the moving loop and opposes the growing upward flux. That is Lenz's law, and nothing had to be added to the formula to get it.

The motional route agrees with the exact value about as well as the Biot-Savart field agrees with the exact field, because it involves no differentiation. The flux route carries the extra differentiation error of the 2 mm step, plus a little amplified quadrature noise.

### A rectangle moving away from a long straight wire

A long straight wire along $z$ carries $I = 5$ A. A rectangular loop lies in the $x$-$z$ plane, with sides $a = 5$ cm along $x$ and $b = 10$ cm along $z$, its near side a distance $s$ from the wire. It moves straight away at $v = 1$ m/s. Here the field is the exact $\mathbf{B} = \mu_0 I\,\hat{\boldsymbol{\phi}}/2\pi r$, so all the error comes from the methods.

The field crosses the loop along $+\hat{\mathbf{y}}$, and integrating $B = \mu_0I/2\pi x$ across the rectangle gives

$$\Phi(s) = \frac{\mu_0 I b}{2\pi}\ln\frac{s + a}{s}, \qquad \mathcal{E} = -\frac{d\Phi}{dt} = \frac{\mu_0 I b}{2\pi}\,\frac{a\,v}{s(s + a)}.$$

The motional emf comes only from the two sides parallel to the wire, where $\mathbf{v}\times\mathbf{B}$ points along them, and it is the difference between the near-side and far-side contributions $vB(s)\,b - vB(s + a)\,b$: the same expression. The rectangle is traversed counter-clockwise about $+\hat{\mathbf{y}}$, which the first printed line confirms. The flux uses a midpoint rule over the rectangle, since it is not a disk.

In [ ]:
I2, aa, bb, v2 = 5.0, 0.05, 0.10, 1.0
B_wire = lambda p: MU0 * I2 / (2 * np.pi) * np.stack([-p[:, 1], p[:, 0], 0 * p[:, 0]], axis=-1) / (p[:, 0] ** 2 + p[:, 1] ** 2)[:, None]

def rectangle(s, n=50):
    corners = np.array([[s, 0, 0], [s, 0, bb], [s + aa, 0, bb], [s + aa, 0, 0]])
    tt = np.linspace(0, 1, n, endpoint=False)[:, None]
    return np.vstack([corners[k] + tt * (corners[(k + 1) % 4] - corners[k]) for k in range(4)])

def rectangle_flux(s, nu=200, nv=50):
    xs = s + (np.arange(nu) + 0.5) * aa / nu
    zs = (np.arange(nv) + 0.5) * bb / nv
    X, Z = np.meshgrid(xs, zs, indexing="ij")
    pts = np.stack([X.ravel(), 0 * X.ravel(), Z.ravel()], axis=-1)
    return np.sum(B_wire(pts)[:, 1]) * aa * bb / (nu * nv)

m, d = bs.segments(rectangle(0.05))
print("orientation, (1/2) sum of r x dl:", 0.5 * np.cross(m - m.mean(axis=0), d).sum(axis=0), "m^2  (area along +y)")

t2 = np.linspace(0.0, 0.1, 101)
s2 = 0.02 + v2 * t2
flux2 = np.array([rectangle_flux(s) for s in s2])
emf2_ex = MU0 * I2 * bb * aa * v2 / (2 * np.pi * s2 * (s2 + aa))
emf2_flux = ind.emf_from_flux(t2, flux2)
emf2_mot = np.array([motional_sum(rectangle(s), np.array([v2, 0, 0]), B_wire) for s in s2])
scale2 = emf2_ex.max()
err2_flux = np.abs(emf2_flux - emf2_ex)[1:-1].max() / scale2
err2_mot = np.abs(emf2_mot - emf2_ex).max() / scale2
print(f"flux vs closed form:      max relative error {np.abs(flux2 / (MU0 * I2 * bb / (2 * np.pi) * np.log((s2 + aa) / s2)) - 1).max():.1e}")
print(f"peak emf at s = 2 cm:     {1e6 * scale2:.3f} uV")
print(f"flux route vs exact:      {err2_flux:.1e} of the peak")
print(f"motional route vs exact:  {err2_mot:.1e} of the peak")

plt.plot(100 * s2, 1e6 * emf2_ex, "k-", lw=3, alpha=0.3, label="closed form")
plt.plot(100 * s2[1:-1], 1e6 * emf2_flux[1:-1], "C0--", label="flux route")
plt.plot(100 * s2, 1e6 * emf2_mot, "C1:", lw=2, label="motional route")
plt.xlabel("distance s of the near side [cm]"); plt.ylabel("emf [uV]"); plt.legend(); plt.title("rectangle moving away from a wire"); plt.show()

Here the motional route is exact to rounding error. On the two moving sides $\mathbf{v}\times\mathbf{B}$ is constant along each side, and on the other two sides it is perpendicular to $d\mathbf{l}$, so the midpoint sum has nothing to approximate. The flux route still carries the time-step error. The emf is positive, so the current runs counter-clockwise about $+\hat{\mathbf{y}}$ and its field adds to the falling flux through the loop. Lenz's law again.

### A magnet falling through a ring

A small bar magnet is a magnetic dipole $\mathbf{m} = m\hat{\mathbf{z}}$ with $m = 1\ \mathrm{A\,m^2}$, roughly a strong neodymium magnet a centimetre long. It falls at a constant $1$ m/s along the axis of a ring of radius $R = 2$ cm. Work in the magnet's frame, where the field is static and the ring moves up at $1$ m/s. The flux through the ring at height $z$ relative to the magnet is

$$\Phi(z) = \frac{\mu_0 m R^2}{2\,(R^2 + z^2)^{3/2}},$$

the flux of a dipole through a coaxial ring (by reciprocity, it is $M$ times the current of an equivalent small loop).

There is a numerical trap here. As the magnet passes through the ring, the flat disk spanning the ring contains the dipole, where the field is infinite, and a midpoint rule over the disk fails. The first lines below show `disk_flux` drifting and then breaking down as the dipole approaches the disk. Any other surface bounded by the ring carries the same flux, because $\nabla\cdot\mathbf{B} = 0$. The cleanest way to use that freedom is Stokes' theorem: $\mathbf{B} = \nabla\times\mathbf{A}$, so $\Phi = \oint\mathbf{A}\cdot d\mathbf{l}$ around the ring itself, which never comes near the dipole. The dipole's vector potential is $\mathbf{A} = \frac{\mu_0}{4\pi}\,\mathbf{m}\times\mathbf{r}/r^3$.

The time window runs from 10 cm below the magnet to 12 cm above it, so the start and end fluxes are both small but not equal.

In [ ]:
m0, R, vd = 1.0, 0.02, 1.0
mvec = np.array([0.0, 0.0, m0])
B_dip = lambda p: bs.dipole_B(mvec, p)
A_dip = lambda p: MU0 / (4 * np.pi) * np.cross(mvec, p) / np.linalg.norm(p, axis=-1, keepdims=True) ** 3
flux_dip = lambda z: MU0 * m0 * R**2 / (2 * (R**2 + z**2) ** 1.5)

print("disk quadrature through the ring, as the dipole approaches the disk:")
for zz in (0.05, 0.01, 0.002, 0.0):
    print(f"  ring {100 * zz:4.1f} cm from the dipole: disk_flux / exact = {ind.disk_flux(B_dip, (0, 0, zz), (0, 0, 1), R, nr=60, nphi=60) / flux_dip(zz):9.4f}")

def flux_by_A(z, n=200):
    mid, dl = bs.segments(bs.loop(R, n, centre=(0, 0, z)))
    return float(np.sum(A_dip(mid) * dl))

t3 = np.linspace(0.0, 0.22, 881)
z3 = -0.10 + vd * t3
flux3 = np.array([flux_by_A(z) for z in z3])
emf3_flux = ind.emf_from_flux(t3, flux3)
emf3_mot = np.array([motional_sum(bs.loop(R, 200, centre=(0, 0, z)), np.array([0, 0, vd]), B_dip) for z in z3])
emf3_ex = 1.5 * MU0 * m0 * R**2 * z3 * vd / (R**2 + z3**2) ** 2.5    # -dPhi/dz * dz/dt
scale3 = np.abs(emf3_ex).max()
err3_A = np.abs(flux3 / flux_dip(z3) - 1).max()
err3_flux = np.abs(emf3_flux - emf3_ex)[1:-1].max() / scale3
err3_mot = np.abs(emf3_mot - emf3_ex).max() / scale3
print(f"flux by the line integral of A vs closed form: max relative error {err3_A:.1e}")
print(f"peak |emf|: {1e3 * scale3:.3f} mV, at z = +-{100 * z3[np.argmax(np.abs(emf3_ex))]:.2f} cm (R/2 = {100 * R / 2:.2f} cm)")
print(f"flux route vs exact:      {err3_flux:.1e} of the peak")
print(f"motional route vs exact:  {err3_mot:.1e} of the peak")

fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].plot(100 * z3, 1e6 * flux3, "C0"); ax[0].set_ylabel("flux [uWb]"); ax[0].set_title("magnet passing through a ring (ring position relative to the magnet)")
ax[1].plot(100 * z3, 1e3 * emf3_ex, "k-", lw=3, alpha=0.3, label="closed form")
ax[1].plot(100 * z3[1:-1], 1e3 * emf3_flux[1:-1], "C0--", label="flux route (A around the ring)")
ax[1].plot(100 * z3, 1e3 * emf3_mot, "C1:", lw=2, label="motional route")
ax[1].axhline(0, color="grey", lw=0.5); ax[1].set_xlabel("z of the ring relative to the magnet [cm]"); ax[1].set_ylabel("emf [mV]"); ax[1].legend()
plt.tight_layout(); plt.show()

The emf has two lobes of opposite sign, separated by zero at the moment the magnet is in the plane of the ring, where the flux peaks. Each lobe peaks at a distance $R/2$ from the ring, where $|d\Phi/dz|$ is largest. As the magnet approaches, the upward flux grows and the induced current makes a downward field that pushes back on the magnet. After it passes, the flux shrinks and the current reverses, making an upward field that pulls the magnet back. Either way the force opposes the motion, which is why a magnet falls slowly through a copper pipe.

The two lobes are mirror images in shape, so their areas nearly cancel: $\int\mathcal{E}\,dt = -[\Phi(t_{\text{end}}) - \Phi(t_{\text{start}})]$, and both end fluxes are small. The section 4 checks make that exact.

### A loop rotating in a uniform field: the generator

A circular loop of radius 5 cm turns at $f = 50$ Hz about the $z$ axis in a uniform field $B_0 = 0.5$ T along $x$. Its normal is $\hat{\mathbf{n}}(t) = (\cos\omega t, \sin\omega t, 0)$, so $\Phi = B_0 A\cos\omega t$ and

$$\mathcal{E} = B_0 A\,\omega\sin\omega t,$$

an alternating emf of amplitude $B_0A\omega$. Here the loop does not translate, so the motional route needs the velocity of each piece of wire, $\mathbf{v} = \boldsymbol{\omega}\times\mathbf{r}$. The helper above accepts a callable velocity for that reason.

In [ ]:
B0, f, Rg = 0.5, 50.0, 0.05
w = 2 * np.pi * f
A_loop = np.pi * Rg**2
B_uni = lambda p: np.tile([B0, 0.0, 0.0], (len(p), 1))
normal = lambda t: np.array([np.cos(w * t), np.sin(w * t), 0.0])

t4 = np.linspace(0.0, 0.04, 801)
flux4 = np.array([ind.disk_flux(B_uni, (0, 0, 0), normal(t), Rg, nr=10, nphi=20) for t in t4])
emf4_flux = ind.emf_from_flux(t4, flux4)
emf4_mot = np.array([motional_sum(bs.loop(Rg, 200, normal=normal(t)), lambda p: np.cross([0.0, 0.0, w], p), B_uni) for t in t4])
emf4_ex = B0 * A_loop * w * np.sin(w * t4)
print(f"amplitude B0 A omega = {B0 * A_loop * w:.4f} V")
print(f"flux route:     max error {np.abs(emf4_flux - emf4_ex)[1:-1].max() / (B0 * A_loop * w):.1e} of the amplitude")
print(f"motional route: max error {np.abs(emf4_mot - emf4_ex).max() / (B0 * A_loop * w):.1e} of the amplitude")

plt.plot(1e3 * t4, emf4_ex, "k-", lw=3, alpha=0.3, label="B0 A omega sin(omega t)")
plt.plot(1e3 * t4[1:-1], emf4_flux[1:-1], "C0--", label="flux route")
plt.plot(1e3 * t4, emf4_mot, "C1:", lw=2, label="motional route")
plt.xlabel("t [ms]"); plt.ylabel("emf [V]"); plt.legend(loc="upper right"); plt.title("a 50 Hz generator"); plt.show()

Here the motional route is limited by the 200-segment polygon. A polygon of $N$ sides inscribed in the circle has an area smaller than the circle's by a fraction $(2\pi/N)^2/6$, which for $N = 200$ is $1.6\times10^{-4}$: the printed error. The flux route integrates over the exact disk, so only the 50 microsecond time step limits it.

### Exercise: the motional emf

Replace the body of `motional_emf` with your own version. It takes a closed polyline `path` of shape $(N, 3)$, a `velocity` that is either one vector (a rigid translation) or a callable mapping points to velocities, and a field `B_func`. It returns $\oint(\mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$ as a midpoint sum. The check applies it to the loop moving along the axis of the current loop, at five positions, against the exact $-I\,v\,dM/dz$.

In [ ]:
def motional_emf(path, velocity, B_func):
    '''Midpoint-rule sum of (v x B) . dl around the closed polyline path.'''
    mid, dl = bs.segments(path)                                       # replace from here with your own version
    vel = velocity(mid) if callable(velocity) else np.broadcast_to(np.asarray(velocity, dtype=float), mid.shape)
    return float(np.einsum("ij,ij->", np.cross(vel, B_func(mid)), dl))


z_test = np.array([-0.15, -0.05, 0.0, 0.03, 0.12])
mine = np.array([motional_emf(bs.loop(b, 200, centre=(0, 0, z)), [0, 0, v], B_loop) for z in z_test])
ref = emf_exact(z_test)
print("z [cm]:        ", 100 * z_test)
print("your emf [uV]: ", 1e6 * mine)
print("exact   [uV]:  ", 1e6 * ref)
assert np.all(np.abs(mine - ref) < 1e-3 * scale1)
print("motional emf exercise passed")

## 4. Verify

Four checks.

1. Both routes agree with the exact emf for the loop on the axis, to the accuracy the time step and the Biot-Savart polygon allow: $2\times10^{-3}$ of the peak for the flux route and $10^{-3}$ for the motional route.
2. For the rectangle beside the wire, the motional route is exact to rounding and the flux route to $2\times10^{-3}$.
3. For the falling magnet, the flux from $\oint\mathbf{A}\cdot d\mathbf{l}$ matches the closed form to $10^{-3}$, and both emf routes match the closed-form emf to $10^{-3}$ of its peak.
4. The time integral of the emf equals minus the total change in flux: $\int\mathcal{E}\,dt = -[\Phi(t_{\text{end}}) - \Phi(t_{\text{start}})]$. This is Faraday's law integrated once, and it holds whatever happens in between. The integral uses the motional emf, which was computed without any reference to the flux, and the trapezoid rule. For the axial loop the flux change is large, because the run ends with the moving loop much closer to the fixed one than it started. For the magnet it is small, because the magnet starts and ends far from the ring, so almost all of the positive lobe is cancelled by the negative one.

In [ ]:
assert err1_flux < 2e-3 and err1_mot < 1e-3
assert err2_mot < 1e-10 and err2_flux < 2e-3
assert err3_A < 1e-3 and err3_flux < 1e-3 and err3_mot < 1e-3

for name, t, emf, flux in (("loop on the axis", t1, emf1_mot, I * M(z1)), ("falling magnet", t3, emf3_mot, flux_dip(z3))):
    integral = np.trapezoid(emf, t)
    change = -(flux[-1] - flux[0])
    scale = np.abs(flux).max()
    print(f"{name:17s}: integral of emf dt = {integral:+.4e} Wb,  -(Phi_end - Phi_start) = {change:+.4e} Wb,  "
          f"difference / max flux = {abs(integral - change) / scale:.1e}")
    assert abs(integral - change) < 2e-4 * scale
print(f"largest positive lobe of the magnet emf, integrated: {np.trapezoid(np.clip(emf3_mot, 0, None), t3):.3e} Wb  (= the peak flux {flux_dip(0.0):.3e} Wb, less the end flux)")

## 5. Explore and rebuild

Explore. Make the falling magnet brake itself. Give the ring a resistance $R_r$, so the induced current is $\mathcal{E}/R_r$ and the power dissipated is $\mathcal{E}^2/R_r$. Energy conservation says that power comes out of the magnet's kinetic energy, so the drag force is $F = \mathcal{E}^2/(R_r v)$. With $\mathcal{E} = -v\,d\Phi/dz$ this is $F = (d\Phi/dz)^2\,v/R_r$: proportional to the speed, like a viscous drag, and strongest at $|z| = R/2$. Write the equation of motion $m_{\text{mag}}\dot v = m_{\text{mag}}g - F$ with $z$ and $v$ as the state, integrate it with `integrators.rk4_ode` for a 10 g magnet dropped from rest 10 cm above a copper ring (choose $R_r$ from the ring's dimensions and copper's resistivity), and compare the loss of kinetic plus potential energy with $\int\mathcal{E}^2/R_r\,dt$. Then stack 50 rings to make a pipe and find the terminal speed. Does it depend on the magnet's mass the way a viscous drag predicts?

Rebuild. Close this notebook. In a blank editor, write a function that takes a loop's position as a function of time and a field, and returns the emf by the flux route: sample the flux with a polar midpoint rule over the spanning disk, then difference it. Write a second function that returns the motional emf at one instant. Reproduce the comparison for the loop on the axis of a current loop and explain, from the size of the time step, why the flux route is the less accurate of the two.